# Diagrama UML

books
- id (PK)
- title
- url
- price
- rating (1–5)
- stock (0–1)
- description
- category_id (FK → categories.id)

authors
- id (PK)
- name (UNIQUE)

book_author
- book_id (FK → books.id)
- author_id (FK → authors.id)

categories
- id (PK)
- name (UNIQUE)

In [ ]:
from bs4 import BeautifulSoup
import requests
from urllib.parse import urljoin
import json, os, time, re, sys
import requests



# Cambio por url de pagina 1 para despues ir cambiando a cada pagina
BASE = "https://books.toscrape.com/"
url = urljoin(BASE, "catalogue/page-1.html")
next_page = url



rating_number = {"One":1,"Two":2,"Three":3,"Four":4,"Five":5}
items = []


OPENLIB_SEARCH = "https://openlibrary.org/search.json"
GOOGLE_BOOKS = "https://www.googleapis.com/books/v1/volumes"

# funcion para obtener autores usando Open Library y Google Books
def get_authors(title, category, lang="en", timeout=2, max_authors=3):
    """
    Devuelve una LISTA de autores usando Open Library y como fallback Google Books.
    Siempre retorna al menos ["Desconocido"] si no encuentra nada.
    """
    def _norm(name: str) -> str:
        return re.sub(r"\s+", " ", name).strip()

    autores = []

    # 1) Open Library (puede traer múltiples)
    try:
        ol = requests.get(
            "https://openlibrary.org/search.json",
            params={"title": title, "limit": 3},
            timeout=timeout
        )
        ol.raise_for_status()
        docs = ol.json().get("docs", [])
        for d in docs:
            for a in d.get("author_name", []) or []:
                a = _norm(a)
                if a and a not in autores:
                    autores.append(a)
                    if len(autores) >= max_authors:
                        break
            if len(autores) >= max_authors:
                break
    except requests.RequestException as e:
        print(f"[⚠️] OpenLibrary error: {e}")

    # 2) Google Books (fallback, agrega si faltan)
    if len(autores) < max_authors:
        query = f'intitle:"{title}"'
        if category:
            query += f' subject:"{category}"'
        try:
            gb = requests.get(
                "https://www.googleapis.com/books/v1/volumes",
                params={
                    "q": query,
                    "maxResults": 3,
                    "orderBy": "relevance",
                    "langRestrict": lang
                },
                timeout=timeout
            )
            gb.raise_for_status()
            items = gb.json().get("items", []) or []
            for it in items:
                for a in it.get("volumeInfo", {}).get("authors", []) or []:
                    a = _norm(a)
                    if a and a not in autores:
                        autores.append(a)
                        if len(autores) >= max_authors:
                            break
                if len(autores) >= max_authors:
                    break
        except requests.RequestException as e:
            print(f"[⚠️] Google Books error: {e}")

    return autores if autores else ["Desconocido"]


# funcion para obtener el soup de una pagina o una url
def get_soup(url_pagina_soup):
    for intento in range(5):  # 🔹 ADICIÓN: reintentos
        try:
            resp = requests.get(url_pagina_soup, timeout=20)
            resp.raise_for_status()
            return BeautifulSoup(resp.text, "lxml")
        except requests.RequestException:
            print(f"[WARN] Falló {url_pagina_soup} (intento {intento+1}): {e}")
            time.sleep(1.2 * (intento + 1))
    raise RuntimeError(f"No pude obtener {url_pagina_soup}")



# mientras existan paginas siguientes, veridicado por boton next. otra forma de hacerlo es iterando
while next_page:

    soup = get_soup(next_page)
    books = soup.find_all("article", class_="product_pod")

    for b in books:


        # Title
        title = b.h3.a["title"]


        # Price
        # 💥💥💥 
        txtprice = b.find("p", class_="price_color").get_text(strip=True)
        price = float(re.sub(r"[^\d.]", "", txtprice))


        # URL
        href = b.h3.a["href"]
        book_url = urljoin(next_page, href)

        try:
            # detalle 
            soup_detail = get_soup(book_url)
        except RuntimeError as e:
            print(f"[SKIP] No pude abrir detalle: {book_url} -> {e}")
            continue   

        # category
        breadcrumb_link = soup_detail.select("ul.breadcrumb li a")
        category = breadcrumb_link[-1].get_text(strip=True) if len(breadcrumb_link) >= 3 else "Unknown"


        # Rating                   
        rating_tag = soup_detail.select_one("p.star-rating")
        classes = rating_tag.get("class", []) if rating_tag else []                        
        rating_word = next((c for c in classes if c in rating_number), "One")
        rating = rating_number.get(rating_word, 1)                         


        # Stock
        stock_text = soup.find("p", class_="instock availability").get_text(strip=True)
        stock_text = stock_text.lower().replace(" ", "")
        stock = 1 if "instock" in stock_text else 0



        # description
        description_tag = soup_detail.find("div", id="product_description")
        description = (
            description_tag.find_next_sibling('p').text.strip()
            if description_tag else None
        )


        # 🔹 ADICIÓN (llamar API): buscar autores por título
        # Explicación:
        # - Usamos la función get_autor(title, category).
        # - Retorna una lista [] con los nombres de los autores o Desconocido.
        # - Se agrega como campo "author" en el item para persistirlo luego en JSON.
        authors = get_authors(title, category)
        # Pequeña pausa de cortesía para no saturar la API si hay muchos libros
        time.sleep(0.15)

        items.append({
            "title": title,
            "category": category,
            "rating": rating,
            "URL": book_url,
            "price": price,
            "stock": stock,
            "description": description,
            # 🔹 ADICIÓN (nuevo campo en el dataset): autores como lista
            # Explicación:
            # - Este campo nuevo te permitirá luego crear tablas 'autores' y 'libro_autor' (M:N) en tu DB.
            # - Mantenerlo como lista te conserva todos los coautores que reporte la API.
            "authors": authors
        })

    time.sleep(0.5)

    # para cada pagina del 1 al 5
    botton_next = soup.find("li", class_="next")
    if botton_next:
        href_next = botton_next.a["href"]
        next_page = urljoin(next_page, href_next)
        time.sleep(0.15)
    else:
        break


# Guardar en JSON
with open('libros_scrapeados.json', 'w', encoding='utf-8') as f:
    json.dump(items, f, ensure_ascii=False, indent=2)

# Leer desde JSON
with open('libros_scrapeados.json', 'r', encoding='utf-8') as f:
    datos = json.load(f)
    print(f'Se guardaron {len(datos)} libros')
    print(datos[0])  # Mostrar el primero para validar estructura

# Abrir automáticamente el archivo (solo en Windows)
os.startfile('libros_scrapeados.json')


#### para limpiar autores con nombres raros y que no son autores

In [2]:
import json, re
from pathlib import Path
from difflib import SequenceMatcher

JSON_PATH = "libros_scrapeados.json"

# 🔹 Palabras que suelen indicar que NO es una persona
BAD_TOKENS = {
    "books", "press", "university", "institute", "guides", "classics",
    "editors", "narrator", "study", "summary", "workbook", "comics",
    "publisher", "worth books"
}

# 🔹 Correcciones manuales de nombres frecuentes
CANON_MAP = {
    "eckart tolle": "Eckhart Tolle",
    "hg wells": "H. G. Wells",
    "h.g.wells": "H. G. Wells",
    "janice y k lee": "Janice Y. K. Lee",
}

# --- Funciones auxiliares ---
def similar(a: str, b: str) -> float:
    """Devuelve una puntuación de similitud (0 a 1) entre dos textos."""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def looks_like_person(name: str) -> bool:
    """Heurísticas para detectar si un texto parece nombre de persona."""
    s = name.strip()
    if not s or len(s) < 2:
        return False
    low = s.lower()
    if any(tok in low for tok in BAD_TOKENS):
        return False
    if re.fullmatch(r"[A-Z0-9\s\.\-]+", s) and s.isupper():
        return False
    # al menos un espacio o punto (iniciales)
    if " " not in s and "." not in s and len(s) < 4:
        return False
    return True

def canon_name(name: str) -> str:
    """Normaliza formato del nombre."""
    n = re.sub(r"\s+", " ", name).strip()
    key = n.lower()
    if key in CANON_MAP:
        return CANON_MAP[key]
    # Espaciado en iniciales, ejemplo: "H.G.Wells" → "H. G. Wells"
    n = re.sub(r"(?<=\w)\.(?=\w)", ". ", n)
    n = re.sub(r"\s+", " ", n)
    # Capitaliza palabras normales
    parts = []
    for p in n.split():
        if p.isupper() and len(p) <= 3:  # iniciales
            parts.append(p)
        else:
            parts.append(p.capitalize())
    return " ".join(parts)

# --- Limpieza principal ---
def limpiar_json(json_path=JSON_PATH, rewrite=True):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    cambios = 0

    for item in data:
        title = item.get("title", "").strip()
        autores = item.get("authors", [])
        nuevos = []

        for a in autores:
            if not isinstance(a, str):
                continue
            a = a.strip()
            if not a:
                continue
            # descartar si el autor es muy parecido al título
            if similar(a, title) > 0.8:
                continue
            # descartar si no parece persona
            if not looks_like_person(a):
                continue
            nuevos.append(canon_name(a))

        # si queda vacío, colocamos 'Desconocido'
        if not nuevos:
            nuevos = ["Desconocido"]

        # eliminar duplicados preservando orden
        vistos = set()
        finales = []
        for n in nuevos:
            key = n.lower()
            if key not in vistos:
                vistos.add(key)
                finales.append(n)

        if finales != autores:
            item["authors"] = finales
            cambios += 1

    if rewrite:
        Path(json_path).write_text(
            json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8"
        )
    print(f"Registros modificados: {cambios}")

# --- Ejecutar ---
if __name__ == "__main__":
    limpiar_json()


Registros modificados: 0


In [9]:
import sqlite3

# 1. Conectamos con la base
# conn representa la conexión a la base de datos.
conn = sqlite3.connect("libros.db")
cursor = conn.cursor()


# 2. Activamos las llaves foráneas (para relaciones entre tablas)
conn.execute("PRAGMA foreign_keys = ON;")   # PRAGMA es una directiva especial de SQLite para configurar opciones




# 3. Escribimos el DDL (definición de tablas)
DDL = """
CREATE TABLE IF NOT EXISTS categories (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS authors (
    id      INTEGER PRIMARY KEY,
    name    TEXT NOT NULL UNIQUE
);

CREATE TABLE IF NOT EXISTS books (
    id          INTEGER PRIMARY KEY,
    title       TEXT NOT NULL,
    url         TEXT NOT NULL UNIQUE,
    price       REAL NOT NULL,
    rating      INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    stock       INTEGER NOT NULL CHECK (stock IN (0,1)),
    description TEXT,
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(id) ON DELETE RESTRICT
);

CREATE TABLE IF NOT EXISTS book_author (
    book_id   INTEGER NOT NULL,
    author_id INTEGER NOT NULL,
    PRIMARY KEY (book_id, author_id),
    FOREIGN KEY (book_id)   REFERENCES books(id)   ON DELETE CASCADE,
    FOREIGN KEY (author_id) REFERENCES authors(id) ON DELETE CASCADE
);
"""

# 4. Ejecutamos todas las sentencias
conn.executescript(DDL)

print("✅ Tablas creadas correctamente")

conn.close()


✅ Tablas creadas correctamente


In [10]:
import sqlite3
conn = sqlite3.connect("libros.db")
cursor = conn.cursor()

#Ver que las tablas se crearon
# SELECT sirve para seleccionar datos de una base de datos
# FROM es para especificar la tabla de donde se obtienen los datos
# sqlite_master es una tabla interna que guarda la estructura de la base
# WHERE sirve para filtrar los resultados
# type='table' filtra solo las filas que representan tablas
cursor.execute('''SELECT name FROM sqlite_master WHERE type='table' ''')
# fetchall() obtiene todas las filas del resultado de la consulta
# fetchall() trae una tupla de todas las filas de la tabla o de la base de datos
tablas = cursor.fetchall()

print("Tablas en la base de datos: ")
print(tablas)

conn.close()

Tablas en la base de datos: 
[('categories',), ('authors',), ('books',), ('book_author',)]


In [ ]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"              # cambia si tu DB se llama distinto
JSON_PATH = "libros_scrapeados.json"  # cambia si tu JSON se llama distinto

def cargar_categorias(db_path=DB_PATH, json_path=JSON_PATH):

    # 1. Cargar JSON
    data = json.loads(Path(JSON_PATH).read_text(encoding="utf-8"))
    print(f"Leídos {len(data)} libros desde el JSON ✅ \n")

    # 2. Conexión + FK on
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 3. recopilar categorías del JSON
    categorias = { item.get("category") for item in data if item.get("category") }
    print(f"categorías : {categorias}")

    # 4. Insertar en la tabla categories
    cur.executemany( "INSERT OR IGNORE INTO categories(name) VALUES (?);", [(c,) for c in sorted(categorias)])

    # 5. Guardar cambios
    conn.commit()

    # 6. Comprobar el resultado
    total = cur.execute("SELECT COUNT(*) FROM categories;").fetchone()[0]
    print(f"Listo categorías. Total en tabla: {total}")
    conn.close()

# Ejecutar solo este paso primero
cargar_categorias()


Leídos 1000 libros desde el JSON ✅ 

categorías : {'Christian', 'Thriller', 'Sequential Art', 'Religion', 'Classics', 'Poetry', 'Music', 'Add a comment', 'History', 'Historical', 'Biography', 'Politics', 'Nonfiction', 'Academic', 'Self Help', 'Childrens', 'Humor', 'Suspense', 'Crime', 'Philosophy', 'Health', 'Christian Fiction', 'Paranormal', 'Psychology', 'Adult Fiction', 'Sports and Games', 'Autobiography', 'Spirituality', 'Parenting', 'Default', 'Novels', 'Young Adult', 'Science', 'Fantasy', 'Horror', 'Food and Drink', 'Womens Fiction', 'Science Fiction', 'New Adult', 'Romance', 'Mystery', 'Erotica', 'Short Stories', 'Contemporary', 'Business', 'Cultural', 'Fiction', 'Travel', 'Art', 'Historical Fiction'}
Listo categorías. Total en tabla: 50


In [12]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"
JSON_PATH = "libros_scrapeados.json"

def cargar_authors(db_path=DB_PATH, json_path=JSON_PATH):
    # 1) Leer JSON -> Python
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))

    # 2) Conexión y FK ON
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 3) Deduplicar autores (siempre lista)
    autores = set()
    for item in data:
        for nm in item["authors"]:
            autores.add(str(nm).strip())

    # 4) Insert masivo sin duplicar en DB
    cur.executemany(
        "INSERT OR IGNORE INTO authors(name) VALUES (?);",
        [(a,) for a in sorted(autores)]
    )

    # 5) Confirmar y mostrar total
    conn.commit()
    total = cur.execute("SELECT COUNT(*) FROM authors;").fetchone()[0]
    print(f"Listo autores. Total en tabla: {total}")
    conn.close()

# Ejecutar
cargar_authors()


Listo autores. Total en tabla: 1469


In [13]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"
JSON_PATH = "libros_scrapeados.json"



def cargar_books(db_path=DB_PATH, json_path=JSON_PATH):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()

    # 1) Obtener mapa categoría nombre->id
    # Precache de categorías name->id para acelerar
    cat_map = dict(cur.execute("SELECT name, id FROM categories;").fetchall())

    # 2) Sentencia de inserción
    insert_sql = """
    INSERT OR IGNORE INTO books(title, url, price, rating, stock, description, category_id)
    VALUES (?, ?, ?, ?, ?, ?, ?);
    """

    # 3) Preparar filas
    filas = []
    for it in data:
        cat = it.get("category")
        cat_id = cat_map.get(cat)

        # Agregar fila, en tupla para la tabla
        # Cada tupla representa una fila a insertar
        filas.append((
            it.get("title"),
            it.get("URL"),
            float(it.get("price") or 0.0),
            int(it.get("rating") or 0),
            int(it.get("stock") or 0),
            it.get("description"),
            cat_id
        ))
        # print(f"Preparando libro: {it.get('title')} (cat_id={cat_id})")

    # 4) Insertar todo
    cur.executemany(insert_sql, filas)
    conn.commit()



    # 5) Confirmar y mostrar total
    total = cur.execute("SELECT COUNT(*) FROM books;").fetchone()[0]
    print(f"Listo books. Total en tabla: {total}")
    conn.close()

# Ejecuta esto como tercer paso
cargar_books()


Listo books. Total en tabla: 1000


In [ ]:
import sqlite3, json
from pathlib import Path

DB_PATH = "libros.db"
JSON_PATH = "libros_scrapeados.json"

def cargar_book_author(db_path=DB_PATH, json_path=JSON_PATH):
    data = json.loads(Path(json_path).read_text(encoding="utf-8"))

    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON;")
    cur = conn.cursor()


    # 1) obtener mapas libro -> id y autor -> id
    # Cache: url libro -> id
    book_map = dict(cur.execute("SELECT url, id FROM books;").fetchall())
    # Cache: autor -> id
    author_map = dict(cur.execute("SELECT name, id FROM authors;").fetchall())


    # 2) sentencia inserción
    insert_rel = "INSERT OR IGNORE INTO book_author(book_id, author_id) VALUES (?, ?);"

    n_rel = 0
    # 3) recorrer datos del JSON 
    for it in data:
        # 4) obtener book_id por URL
        url = it.get("URL")
        book_id = book_map.get(url)

        # 5) obtener lista de autores
        autores = it.get("authors") or []

        # 6) iterar autores 
        for a in autores:
            # limpiar nombre
            a = str(a).strip()
            # obtener author_id
            a_id = author_map.get(a)

            # insertar relación
            cur.execute(insert_rel, (book_id, a_id))
            n_rel += 1

    # 7) confirmar y mostrar totales
    conn.commit()
    total_rel = cur.execute("SELECT COUNT(*) FROM book_author;").fetchone()[0]
    print(f"Listo relaciones. Agregadas ahora: {n_rel}. Total en tabla: {total_rel}")
    conn.close()

# Ejecuta esto como cuarto paso
cargar_book_author()


Listo relaciones. Agregadas ahora: 2005. Total en tabla: 2005


# Consultas

In [4]:
import sqlite3, pandas as pd
def run(sql, params=()):
    conn = sqlite3.connect("libros.db")
    display(pd.read_sql_query(sql, conn, params=params))
    conn.close()


In [43]:
# 1. Libros con más de 3 estrellas y a menos de £12
%time
query = """
SELECT title, price, rating
FROM books
WHERE rating > 3 AND price < 12;
"""
run(query)

# SELECT sirve para seleccionar datos de una base de datos.
# FROM es para especificar la tabla de donde se obtienen los datos.
# WHERE sirve para filtrar los resultados según una condición.


CPU times: total: 0 ns
Wall time: 12.4 μs


,title,price,rating
0,Superman Vol. 1: Before Truth (Superman by Gen...,11.89,5
1,Old School (Diary of a Wimpy Kid #10),11.83,5
2,I Am Pilgrim (Pilgrim #1),10.60,4
3,Greek Mythic History,10.23,5
4,Dear Mr. Knightley,11.21,5
5,City of Fallen Angels (The Mortal Instruments #4),11.23,4
6,"The Sleep Revolution: Transforming Your Life, ...",11.68,4
7,"NaNo What Now? Finding your editing process, r...",10.41,4
8,History of Beauty,10.29,4
9,The Origin of Species,10.01,4


In [ ]:
# 2. Libros con precio > 50
%time
query = """
SELECT title, price
FROM books
WHERE price > 50
Limit 5;
"""
run(query)

# LIMIT sirve para limitar la cantidad de resultados devueltos por una consulta.


CPU times: total: 0 ns
Wall time: 15.5 μs


,title,price
0,A Light in the Attic,51.77
1,Tipping the Velvet,53.74
2,Soumission,50.10
3,Sapiens: A Brief History of Humankind,54.23
4,The Black Maria,52.15


In [ ]:
# 3. Autores con más libros publicados (top 10)
%time
query = """
SELECT a.name AS author, COUNT(ba.book_id) AS num_books
FROM authors a
JOIN book_author ba ON a.id = ba.author_id
GROUP BY a.id
ORDER BY num_books DESC
LIMIT 10;
"""
run(query)

# JOIN se utiliza para combinar filas de dos o más tablas basándose en una columna relacionada entre ellas.
# GROUP BY se utiliza para agrupar filas que tienen los mismos valores en columnas especificadas.
# ORDER BY se utiliza para ordenar los resultados de una consulta en orden ascendente o descendente.
# AS se utiliza para asignar un alias a una columna o tabla en una consulta SQL.


CPU times: total: 0 ns
Wall time: 11.4 μs


,author,num_books
0,Desconocido,373
1,Stephen King,9
2,Sir James Augustus Henry Murray,6
3,J. K. Rowling,5
4,Irb Media,5
5,David Levithan,5
6,Sophie Kinsella,4
7,Robert Lawrence Stine,4
8,Instaread,4
9,Gillian Flynn,4


In [ ]:
# 4. categorias con más de 20 libros
%time
query = """
SELECT c.name, COUNT(*) AS total_libros
FROM categories c
JOIN books b ON c.id = b.category_id
GROUP BY c.id
HAVING COUNT(*) > 20
ORDER BY total_libros DESC;
"""
run(query)

# HAVING se utiliza para filtrar grupos de resultados después de aplicar una función de agregación, como COUNT o SUM.


CPU times: total: 0 ns
Wall time: 17.2 μs


,name,total_libros
0,Default,152
1,Nonfiction,110
2,Sequential Art,75
3,Add a comment,67
4,Fiction,65
5,Young Adult,54
6,Fantasy,48
7,Romance,35
8,Mystery,32
9,Food and Drink,30


In [46]:
# 5. Libros que mencionan 'mystery' en la descripción
%time
query = """
SELECT title, description
FROM books
WHERE description LIKE '%mystery%'
LIMIT 5;
"""
run(query)

# LIKE se utiliza en una cláusula WHERE para buscar un patrón específico en una columna de texto.


CPU times: total: 0 ns
Wall time: 13.4 μs


,title,description
0,In Her Wake,A perfect life â¦ until she discovered it was...
1,The Past Never Ends,"A simple task, Attorney Chester Morgan thinks...."
2,orange: The Complete Collection 1 (orange: The...,A Plea From the FutureOn the day that Naho beg...
3,"Lumberjanes, Vol. 2: Friendship to the Max (Lu...","What a mystery!Jo, April, Mal, Molly, and Ripl..."
4,"Lumberjanes, Vol. 1: Beware the Kitten Holy (L...",FRIENDSHIP TO THE MAX!At Miss Qiunzilla Thiskw...


In [ ]:
# 6. Mejores autores por calificación promedio con cantidad minima de libros
%time
query = """
SELECT a.name AS author, AVG(b.rating) AS avg_rating, COUNT(ba.book_id) AS num_books
FROM authors AS a
JOIN book_author AS ba ON a.id = ba.author_id
JOIN books AS b ON ba.book_id = b.id
GROUP BY a.id
HAVING COUNT(ba.book_id) >= 5
ORDER BY avg_rating DESC
LIMIT 10;
"""
run(query)

# AVG se utiliza para calcular el valor promedio de una columna numérica en un conjunto de resultados.



CPU times: total: 0 ns
Wall time: 16.7 μs


,author,avg_rating,num_books
0,Irb Media,3.000000,5
1,Desconocido,2.927614,373
2,Stephen King,2.888889,9
3,J. K. Rowling,2.800000,5
4,David Levithan,2.800000,5
5,Sir James Augustus Henry Murray,2.000000,6


ERROR! Session/line number was not unique in database. History logging moved to new session 127


In [12]:
query = """
SELECT libros_baratos.title, libros_baratos.price, libros_baratos.rating
FROM (SELECT * FROM books WHERE price < 20) AS libros_baratos
WHERE libros_baratos.rating >= 4.5;
"""

run(query)

,title,price,rating
0,Set Me Free,17.46,5
1,The Four Agreements: A Practical Guide to Pers...,17.66,5
2,Sophie's World,15.94,5
3,Thirst,17.27,5
4,"Princess Jellyfish 2-in-1 Omnibus, Vol. 01 (Pr...",13.61,5
5,Princess Between Worlds (Wide-Awake Princess #5),13.34,5
6,The Third Wave: An Entrepreneurâs Vision of ...,12.61,5
7,Dark Notes,19.19,5
8,Batman: The Dark Knight Returns (Batman),15.38,5
9,Agnostic: A Spirited Manifesto,12.51,5


In [ ]:
query = """

"""
run(query)